In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
def load_wesad_data(directory):
    data = {}
    for participant_id in os.listdir(directory):
        participant_path = os.path.join(directory, participant_id)
        if os.path.isdir(participant_path):
            participant_data = {}
            for file_name in os.listdir(participant_path):
                file_path = os.path.join(participant_path, file_name)
                if file_name.endswith('.pkl'):
                    participant_data[file_name.split('.')[0]] = pd.read_pickle(file_path)
            data[participant_id] = participant_data
    return data

wesad_data = load_wesad_data('C:/Users/rusha/Desktop/Uni_Freiburg_Notes/MDD/WESAD')

In [2]:
def preprocess_data(wesad_data):
    processed_data = {}
    for participant_id, participant_data in wesad_data.items():
        processed_participant_data = {}
        for sensor, data in participant_data.items():
            if isinstance(data, pd.DataFrame):
                data = data.fillna(data.mean())
                data = (data - data.min()) / (data.max() - data.min())
                processed_participant_data[sensor] = data
        processed_data[participant_id] = processed_participant_data
    return processed_data

preprocessed_wesad_data = preprocess_data(wesad_data)

In [3]:
labels = wesad_data["S10"]["S10"]["label"]
print(Counter(labels).keys()) # equals to list(set(words))
print(Counter(labels).values())

dict_keys([0, 1, 5, 3, 7, 4, 2, 6])
dict_values([1589000, 826000, 35700, 260400, 39900, 557200, 507500, 31500])


In [ ]:
'''IGNORE FOR NOW'''


# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report

# # Split data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# # Train the Random Forest classifier
# model = RandomForestClassifier(n_estimators=100, random_state=42)
# model.fit(X_train, y_train)

# # Predict on the test set
# y_pred = model.predict(X_test)

# # Evaluate the model
# accuracy = accuracy_score(y_test, y_pred)
# print(f'Accuracy: {accuracy}')
# print(classification_report(y_test, y_pred))

In [4]:
# subject 's10'
subject_s10_data = wesad_data['S10']

print("Structure of data for subject S10:")
s = subject_s10_data['S10']['signal']['chest']['Temp']
print(s.shape)


Structure of data for subject S10:
(3847200, 1)


In [167]:
time_original = np.linspace(0, len(s) / 64, len(s))
print(len(time_original))
time_target = np.linspace(0, len(s) / 64, 3847200)
print(len(time_target))
time_target = time_target.reshape(-1, 1)
time_original = time_original.reshape(-1, 1)

351744
3847200


In [176]:
print(time_target.shape)
print(time_original.shape)
print(s.shape)

(3847200, 1)
(351744, 1)
(351744, 1)


In [175]:
y_new = np.interp(time_target, time_original, s)

ValueError: object too deep for desired array

In [5]:
subject_s10_data = wesad_data['S10']
s10_data_details = subject_s10_data['S10']
labels_s10 = s10_data_details['label']
print("labels for subject S10:")
print(labels_s10)

labels for subject S10:
[0 0 0 ... 0 0 0]


In [6]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def extract_and_process_data(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    chest_data = subject_details['signal']['chest']
    labels = subject_details['label']
    
    # flatten and reshape each signal
    ecg = chest_data['ECG'].flatten().reshape(-1, 1)
    emg = chest_data['EMG'].flatten().reshape(-1, 1)
    eda = chest_data['EDA'].flatten().reshape(-1, 1)
    temp = chest_data['Temp'].flatten().reshape(-1, 1)
    resp = chest_data['Resp'].flatten().reshape(-1, 1)
    
    # print the shapes of all features
    print(f"ECG shape: {ecg.shape}")
    print(f"EMG shape: {emg.shape}")
    print(f"EDA shape: {eda.shape}")
    print(f"TEMP shape: {temp.shape}")
    print(f"RESP shape: {resp.shape}")
    
    # combine the features into a single array
    features = np.hstack((
        ecg,
        emg,
        eda,
        temp,
        resp
    ))

    return features, labels

# extract and process data from subjects S10 and S11
features_s10, labels_s10 = extract_and_process_data('S10')
features_s11, labels_s11 = extract_and_process_data('S11')

# combine features and labels from both subjects
combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))

l = pd.DataFrame(np.hstack((combined_features, combined_labels.reshape(-1, 1))))

ECG shape: (3847200, 1)
EMG shape: (3847200, 1)
EDA shape: (3847200, 1)
TEMP shape: (3847200, 1)
RESP shape: (3847200, 1)
ECG shape: (3663100, 1)
EMG shape: (3663100, 1)
EDA shape: (3663100, 1)
TEMP shape: (3663100, 1)
RESP shape: (3663100, 1)


In [7]:
l = l[l[5] != 5]
l = l[l[5] != 6]
l = l[l[5] != 7]

In [8]:
combined_features = l.loc[:,:4]
combined_labels = l.loc[:, 5]
print(combined_features)
print(combined_labels)


                0         1         2          3         4
0       -1.333694 -0.013687  0.716019  33.695862  0.213623
1       -1.327744 -0.021927  0.714493  33.741333  0.192261
2       -1.322067 -0.009018  0.715637  33.717072  0.205994
3       -1.316345 -0.002380  0.714874  33.741333  0.193787
4       -1.310257  0.001053  0.715256  33.747406  0.172424
...           ...       ...       ...        ...       ...
7510295 -0.027145 -0.045914  6.463242  35.029724 -0.650024
7510296 -0.032822 -0.047928  6.377792  34.989563 -0.669861
7510297 -0.026779 -0.045502  6.378174  35.014282 -0.639343
7510298 -0.006821 -0.041977  6.377029  35.018921 -0.648499
7510299  0.016434 -0.020828  6.380081  35.023529 -0.682068

[7296801 rows x 5 columns]
0          0.0
1          0.0
2          0.0
3          0.0
4          0.0
          ... 
7510295    0.0
7510296    0.0
7510297    0.0
7510298    0.0
7510299    0.0
Name: 5, Length: 7296801, dtype: float64


In [10]:
# check initial label distribution
initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

X_train, X_test, y_train, y_test = train_test_split(combined_features, combined_labels, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=10, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# check the label distribution in  training set
label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

# check the label distribution in test set
label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)


Initial label distribution: Counter({0.0: 3032400, 1.0: 1652000, 4.0: 1110901, 2.0: 983500, 3.0: 518000})
Accuracy: 0.9337141392705437
Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.93      0.93    607025
         1.0       0.99      0.99      0.99    330537
         2.0       0.96      0.94      0.95    196988
         3.0       0.89      0.87      0.88    103660
         4.0       0.89      0.87      0.88    221151

    accuracy                           0.93   1459361
   macro avg       0.93      0.92      0.93   1459361
weighted avg       0.93      0.93      0.93   1459361

Label distribution in the training set: Counter({0.0: 2425375, 1.0: 1321463, 4.0: 889750, 2.0: 786512, 3.0: 414340})
Label distribution in the test set: Counter({0.0: 607025, 1.0: 330537, 4.0: 221151, 2.0: 196988, 3.0: 103660})


In [137]:
len(s10_data_details["signal"]["wrist"]["TEMP"])

21984

In [122]:
features_s10 = s10_data_details['signal']['wrist']['ACC']
labels_s10 = s10_data_details['label']

In [123]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_s10_normalized = scaler.fit_transform(features_s10)

In [124]:
print("Shape of features_s10:", features_s10.shape)
print("Shape of labels_s10:", labels_s10.shape)

Shape of features_s10: (175872, 3)
Shape of labels_s10: (3847200,)


In [126]:
print(Counter(labels_s10).keys()) # equals to list(set(words))
print(Counter(labels_s10).values())

dict_keys([0, 1, 5, 3, 7, 4, 2, 6])
dict_values([1589000, 826000, 35700, 260400, 39900, 557200, 507500, 31500])


In [127]:
# ensure number of rows in features_s10 matches no. of labels
n_samples = min(features_s10.shape[0], len(labels_s10))

# trim features and labels to have same no. of samples
features_s10_trimmed = features_s10[:n_samples]
labels_s10_trimmed = labels_s10[:n_samples]

X_train, X_test, y_train, y_test = train_test_split(features_s10_trimmed, labels_s10_trimmed, test_size=0.2, random_state=42)

In [115]:
y_train

array([0, 0, 0, ..., 0, 0, 0])

In [128]:
print(Counter(y_train).keys()) # equals to list(set(words))
print(Counter(y_train).values())

dict_keys([0, 1])
dict_values([52090, 88607])


In [54]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"accuracy: {accuracy:.2f}")
print("classification report:")
print(report)

accuracy: 0.94
classification report:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92     12910
           1       0.95      0.95      0.95     22265

    accuracy                           0.94     35175
   macro avg       0.94      0.94      0.94     35175
weighted avg       0.94      0.94      0.94     35175



In [82]:
import numpy as np
import pandas as pd

# extract the wrist data and labels for subject S10
subject_s10_data = wesad_data['S11']
s10_data_details = subject_s10_data['S11']

wrist_data = s10_data_details['signal']['wrist']
labels_s10 = s10_data_details['label']

# determine the minimum length
min_length = min(
    wrist_data['ACC'].shape[0],
    wrist_data['BVP'].shape[0],
    wrist_data['EDA'].shape[0],
    wrist_data['TEMP'].shape[0]
)

# trim all arrays to minimum length
acc_trimmed = wrist_data['ACC'][:min_length]
bvp_trimmed = wrist_data['BVP'][:min_length]
eda_trimmed = wrist_data['EDA'][:min_length]
temp_trimmed = wrist_data['TEMP'][:min_length]

# combine wrist data components into a single DataFrame
features_s10 = np.hstack((acc_trimmed, bvp_trimmed, eda_trimmed, temp_trimmed))

# ensure that the number of rows in features_s10 matches the number of labels
n_samples = min(features_s10.shape[0], len(labels_s10))

# trim features and labels to have the same number of samples
features_s10_trimmed = features_s10[:n_samples]
labels_s10_trimmed = labels_s10[:n_samples]


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_s10_normalized = scaler.fit_transform(features_s10_trimmed)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(features_s10_normalized, labels_s10_trimmed, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


clf = RandomForestClassifier(n_estimators=100, random_state=42)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)


Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4187

    accuracy                           1.00      4187
   macro avg       1.00      1.00      1.00      4187
weighted avg       1.00      1.00      1.00      4187



In [83]:
import numpy as np
from collections import Counter

# Check class distribution in the dataset
label_counts = Counter(labels_s10_trimmed)
print("Label distribution:", label_counts)

Label distribution: Counter({0: 20932})


In [138]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def extract_and_process_data(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    wrist_data = subject_details['signal']['wrist']
    labels = subject_details['label']

    # ensuring labels are within a valid range
    valid_labels = np.isin(labels, [0, 1, 2, 3, 4])
    labels = labels[valid_labels]
    
    # downsample the features to match the labels' length
    def downsample(signal, target_length):
        return signal[:target_length] 

    num_samples = len(labels)

    features = np.hstack((
        downsample(wrist_data['ACC'], num_samples),
        downsample(wrist_data['BVP'], num_samples),
        downsample(wrist_data['EDA'], num_samples),
        downsample(wrist_data['TEMP'], num_samples)
    ))

    return features, labels

features_s10, labels_s10 = extract_and_process_data('S10')
features_s11, labels_s11 = extract_and_process_data('S11')

combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))

initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

valid_labels = np.isin(combined_labels, [0, 1, 2, 3, 4])
balanced_features = combined_features[valid_labels]
balanced_labels = combined_labels[valid_labels]


min_samples_per_class = min(Counter(balanced_labels).values())

balanced_features_list = []
balanced_labels_list = []

for label in np.unique(balanced_labels):
    label_indices = np.where(balanced_labels == label)[0]
    sampled_indices = np.random.choice(label_indices, min_samples_per_class, replace=False)
    balanced_features_list.append(balanced_features[sampled_indices])
    balanced_labels_list.append(balanced_labels[sampled_indices])

# Convert the balanced features and labels to numpy arrays
balanced_features = np.vstack(balanced_features_list)
balanced_labels = np.hstack(balanced_labels_list)


balanced_label_distribution = Counter(balanced_labels)
print("Balanced label distribution:", balanced_label_distribution)

X_train, X_test, y_train, y_test = train_test_split(balanced_features, balanced_labels, test_size=0.2, random_state=42, stratify=balanced_labels)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 175872 and the array at index 1 has size 351744

In [87]:

# Extract and combine data from subjects S10 and S11
features_s10, labels_s10 = extract_wrist_data_labels('S10')
features_s11, labels_s11 = extract_wrist_data_labels('S11')

combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))


initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)




Initial label distribution: Counter({0: 42916})
